In [0]:
df = spark.read.csv("/Volumes/workspace/default/project/Reseller.csv", sep=[",/t"] , header=True , inferSchema=True)

display(df)

## All the values are in the fisrt column . Only one column exists

In [0]:
df.describe()

In [0]:
df.printSchema()

## Distribute values in the multiple columns

In [0]:
from pyspark.sql.functions import split, col, trim

#
raw_df = spark.read.text("/Volumes/workspace/default/project/Reseller.csv")

# 2. Χρησιμοποιούμε regex στο split: το [,\t]+ σημαίνει "ένα ή περισσότερα κόμματα ή tabs"
# Αυτό θα δημιουργήσει μια στήλη τύπου array
df_split = raw_df.withColumn("cols", split(col("value"), r"[,\t]+"))

# 3. Επιλέγουμε τα στοιχεία του array και τα αντιστοιχίζουμε στις στήλες σου
# Χρησιμοποιούμε trim() για να καθαρίσουμε τυχόν εναπομείναντα κενά
final_df = df_split.select(
    trim(col("cols").getItem(0)).alias("ResellerKey"),
    trim(col("cols").getItem(1)).alias("Business Type"),
    trim(col("cols").getItem(2)).alias("Reseller"),
    trim(col("cols").getItem(3)).alias("City"),
    trim(col("cols").getItem(4)).alias("State-Province"),
    trim(col("cols").getItem(5)).alias("Country-Region")
)

# 4. Εμφάνιση αποτελέσματος
display(final_df)

## Columns creation and data type definitions

In [0]:
from pyspark.sql.types import StructType, StructField, StringType , IntegerType

schema = StructType([
    StructField("ResellerKey", IntegerType(), True),
    StructField("Business Type", StringType(), True),
    StructField("Reseller", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State-Province", StringType(), True),
    StructField("Country-Region", StringType(), True),
])



In [0]:
from pyspark.sql.functions import col

# 1. Αφαιρούμε τη γραμμή που περιέχει τους τίτλους των στηλών
final_df = final_df.filter(col("ResellerKey") != "ResellerKey")

for field in schema:
    final_df = final_df.withColumn(field.name, col(field.name).cast(field.dataType))

display(final_df)